In [1]:
import sys
import os
import torch

In [2]:
# from federated_learning_cuda import load_configuration, main_experiment
from run import load_configuration, main_experiment

/home/brian_bosho/FP/FP/federated-gnn/old_src/utils.py:58: SyntaxWarning: invalid escape sequence '\m'
  """


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   
from run import load_and_split_with_khop, load_and_split_with_feature_prop
dataset_name = "Cora"
num_clients = 10
beta = 1
hop = 1
data_loading_option = "full"
fulltraining_flag = False



In [4]:
cora_data = load_and_split_with_khop(dataset_name, device, num_clients, beta, hop=hop, imputation_method=data_loading_option, fulltraining_flag=fulltraining_flag)

In [5]:
len(cora_data)

4

In [6]:
# # check client 0 data
# print(clients_data[0])
# # check number of nodes in client 0 with zerof feature vectors
# # also check train/test/val nodes the number
# #total nodes in client 0
# total_nodes = clients_data[0].x.shape[0]
# #nodes with zero feature vectors
# zfv_nodes = torch.sum(clients_data[0].x == 0)
# print(f"Number of nodes in client 0 with zero feature vectors: {torch.sum(clients_data[0].x == 0)}")
# print(f"Number of train nodes in client 0: {torch.sum(clients_data[0].train_mask)}")
# print(f"Number of test nodes in client 0: {torch.sum(clients_data[0].test_mask)}")
# print(f"Number of val nodes in client 0: {torch.sum(clients_data[0].val_mask)}")




In [7]:
# load with fp and check the same things
data, dataset, clients_data, test_data,  = load_and_split_with_feature_prop("Citeseer", device, num_clients=10, beta=0.5, hop=1)

# check client 0 data
print(clients_data[0])
# check number of nodes in client 0 with zerof feature vectors
print(f"Number of nodes in client 0 with zero feature vectors: {torch.sum(clients_data[0].x == 0)}")
print(f"Number of train nodes in client 0: {torch.sum(clients_data[0].train_mask)}")
print(f"Number of test nodes in client 0: {torch.sum(clients_data[0].test_mask)}")
print(f"Number of val nodes in client 0: {torch.sum(clients_data[0].val_mask)}")
    


Data(x=[936, 3703], edge_index=[2, 2454], y=[936], train_mask=[936], val_mask=[936], test_mask=[936], mapping=[339])
Number of nodes in client 0 with zero feature vectors: 3273943
Number of train nodes in client 0: 7
Number of test nodes in client 0: 104
Number of val nodes in client 0: 55


In [8]:
import torch

print(f"Number of GPUs available: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("No GPU available")

Number of GPUs available: 1
GPU 0: NVIDIA L40-48Q


In [ ]:
# from federated_learning_cuda import load_configuration, main_experiment
from run import load_configuration, main_experiment
import os
import ray
ray.shutdown()

if __name__ == "__main__":
    # Define all options in a list
    data_loading_options = ["full", "adjacency", "diffusion", ]
    model_types = ["GCN"]
        
    # Load configuration once since it's common for all runs
    clients_num, beta, cfg = load_configuration()

    beta = 10000

    clients_num = 10

    # Main results directory
    main_dir = 'Central_results474'
    dataset_name = "Cora"  # You can change this if you have different datasets

    # Loop over all combinations of data_loading_option and model_type
    for data_loading_option in data_loading_options:
        for model_type in model_types:
            # Create a structured directory for each experiment
            result_name = f"{dataset_name}_{data_loading_option}_{model_type}"
            results_dir = os.path.join(main_dir, result_name)

            # Create the directory if it doesn't exist
            os.makedirs(results_dir, exist_ok=True)

            print(f"Running experiment with data_loading_option: {data_loading_option}, model_type: {model_type}")
            result, output = main_experiment(clients_num, beta, data_loading_option, model_type, cfg, dataset_name=dataset_name, hop=1)
            
            # Replace literal '\n' with actual newline characters if necessary
            # Convert result dictionary to string representation first
            result_str = str(result)
            result_with_newlines = result_str.replace('\\n', '\n')
            
            # Create a unique filename for each combination
            filename = f'results_{dataset_name}_{data_loading_option}_{model_type}.txt'
            filepath = os.path.join(results_dir, filename)
            
            # Write the result to the text file
            with open(filepath, 'w') as f:
                f.write(result_with_newlines)
            
            print(f"Results saved to {filepath}\n")

Running experiment with data_loading_option: full, model_type: GCN
DEVICE: cuda
Data loading option is full
Model type is GCN


2025-05-02 16:59:39,080	INFO worker.py:1816 -- Started a local Ray instance.


data_loading_option: full
Data loaded
Device is cuda
Cora()
10
(FLClient pid=2660231) Training model: <class 'gnn_models.GCN'>


(FLClient pid=2660231) 2025-05-02 16:59:45,136 - INFO - Epoch   0| Train Loss: 1.947| Train Accuracy: 0.071
(FLClient pid=2660231) 2025-05-02 16:59:45,140 - INFO - Epoch   1| Train Loss: 1.904| Train Accuracy: 0.357
(FLClient pid=2660231) 2025-05-02 16:59:45,143 - INFO - Epoch   2| Train Loss: 1.852| Train Accuracy: 0.357
(FLClient pid=2660233) 2025-05-02 16:59:45,194 - INFO - Epoch   2| Validation Loss: 2.030, Validation Accuracy: 0.105


(FLClient pid=2660231) Training model: <class 'gnn_models.GCN'> [repeated 140x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)


...

(FLClient pid=2664539) Training model: <class 'gnn_models.GCN'> [repeated 160x across cluster]


(FLClient pid=2664539) 2025-05-02 17:01:40,682 - INFO - Epoch   2| Train Loss: 1.665| Train Accuracy: 0.312 [repeated 481x across cluster]
(FLClient pid=2664539) 2025-05-02 17:01:40,686 - INFO - Epoch   2| Validation Loss: 2.127, Validation Accuracy: 0.105 [repeated 161x across cluster]


Training done
Round 1
Train Loss: 2.011, Train Accuracy: 0.429
Train Loss: 2.006, Train Accuracy: 0.312
Train Loss: 2.045, Train Accuracy: 0.267
Train Loss: 1.851, Train Accuracy: 0.300
Train Loss: 1.931, Train Accuracy: 0.357
Train Loss: 1.879, Train Accuracy: 0.267
Train Loss: 1.975, Train Accuracy: 0.267
Train Loss: 1.995, Train Accuracy: 0.214
Train Loss: 1.961, Train Accuracy: 0.286
Train Loss: 1.916, Train Accuracy: 0.231
Round 2
Train Loss: 2.024, Train Accuracy: 0.214
Train Loss: 2.023, Train Accuracy: 0.312
Train Loss: 2.042, Train Accuracy: 0.200
Train Loss: 1.853, Train Accuracy: 0.300
Train Loss: 1.932, Train Accuracy: 0.357
Train Loss: 1.850, Train Accuracy: 0.267
Train Loss: 1.977, Train Accuracy: 0.267
Train Loss: 2.015, Train Accuracy: 0.214
Train Loss: 1.958, Train Accuracy: 0.214
Train Loss: 1.898, Train Accuracy: 0.154
Round 3
Train Loss: 2.030, Train Accuracy: 0.357
Train Loss: 2.036, Train Accuracy: 0.312
Train Loss: 2.051, Train Accuracy: 0.200
Train Loss: 1.845, 

(FLClient pid=2664556) 2025-05-02 17:01:41,995 - INFO - Epoch   2| Train Loss: 1.703| Train Accuracy: 0.385 [repeated 147x across cluster]
(FLClient pid=2664556) 2025-05-02 17:01:41,997 - INFO - Epoch   2| Validation Loss: 1.905, Validation Accuracy: 0.133 [repeated 49x across cluster]


Results saved to Central_results454/Cora_diffusion_GCN/results_Cora_diffusion_GCN.txt



In [10]:
import os
import ast # To safely evaluate string literal as dictionary
import pandas as pd
import logging # Using logging for better error reporting

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def process_results_folder(main_folder_path):
    """
    Processes experiment results stored in subfolders of a main directory.

    Reads .txt files containing dictionary-like strings, parses them,
    and organizes the configuration and summary data into a Pandas DataFrame.

    Args:
        main_folder_path (str): The path to the main results folder
                                (e.g., 'Central_results2').

    Returns:
        pandas.DataFrame: A DataFrame containing the structured results,
                          with each row representing an experiment.
                          Returns an empty DataFrame if the folder is not found
                          or no valid results are processed.
    """
    all_results_data = [] # List to hold data for each valid experiment

    # --- Input Validation ---
    if not os.path.isdir(main_folder_path):
        logging.error(f"Folder not found at '{main_folder_path}'")
        return pd.DataFrame() # Return empty DataFrame

    logging.info(f"Starting processing for folder: '{main_folder_path}'")

    # --- Iterate Through Subfolders ---
    try:
        subfolder_names = os.listdir(main_folder_path)
    except OSError as e:
        logging.error(f"Could not list directory contents for '{main_folder_path}': {e}")
        return pd.DataFrame()

    for item_name in subfolder_names:
        subfolder_path = os.path.join(main_folder_path, item_name)

        # Check if it's actually a directory
        if not os.path.isdir(subfolder_path):
            logging.debug(f"Skipping non-directory item: '{item_name}'")
            continue

        logging.info(f"Processing subfolder: '{item_name}'")

        # --- Find the .txt File ---
        txt_file_path = None
        try:
            files_in_subfolder = os.listdir(subfolder_path)
            for filename in files_in_subfolder:
                if filename.endswith(".txt"):
                    # Assume only one relevant .txt file per subfolder
                    txt_file_path = os.path.join(subfolder_path, filename)
                    logging.debug(f"Found results file: '{filename}'")
                    break # Stop searching in this subfolder once found
            if not txt_file_path:
                 logging.warning(f"No .txt file found in subfolder '{item_name}'. Skipping.")
                 continue
        except OSError as e:
            logging.warning(f"Could not list files in subfolder '{item_name}': {e}. Skipping.")
            continue

        # --- Read and Parse File Content ---
        try:
            with open(txt_file_path, 'r', encoding='utf-8') as f:
                content = f.read()

            # Safely parse the string content into a Python dictionary
            # ast.literal_eval is safer than eval() for untrusted input
            data = ast.literal_eval(content)

            if not isinstance(data, dict):
                 logging.warning(f"Parsed content from '{txt_file_path}' is not a dictionary. Skipping.")
                 continue

        except FileNotFoundError:
            # This might happen in rare race conditions if the file is deleted
            # between listing and opening
            logging.warning(f"File '{txt_file_path}' found initially but could not be opened. Skipping.")
            continue
        except (SyntaxError, ValueError) as e:
            logging.warning(f"Error parsing content in file '{txt_file_path}': {e}. Skipping.")
            continue
        except Exception as e:
            logging.error(f"An unexpected error occurred reading/parsing '{txt_file_path}': {e}. Skipping.")
            continue

        # --- Extract Data ---
        # Use .get() for safer access in case keys are missing
        config = data.get('experiment_config', {})
        summary = data.get('summary', {})

        if not config or not summary:
             logging.warning(f"Missing 'experiment_config' or 'summary' key in '{txt_file_path}'. Skipping file.")
             continue

        # Create a flat dictionary for the DataFrame row
        # Start with configuration parameters
        row_data = config.copy()

        # Add summary results, prefixing keys if needed to avoid clashes (optional)
        # Example: prefix summary keys with 'summary_'
        # summary_prefixed = {f'summary_{k}': v for k, v in summary.items()}
        # row_data.update(summary_prefixed)

        # Or just update directly if keys are distinct
        row_data.update(summary)

        # Add an identifier for the experiment (using subfolder name)
        row_data['experiment_id'] = item_name
        # You could also add the filename if needed:
        # row_data['source_file'] = os.path.basename(txt_file_path)

        # Remove potentially large list/dict columns if not needed directly in the summary table
        # Adjust these based on what you want in the final table
        row_data.pop('global_results', None) # Remove list of all global results per round
        row_data.pop('client_results', None) # Remove list of all client results per round
        row_data.pop('rounds', None)         # Remove the detailed rounds data

        all_results_data.append(row_data)
        logging.debug(f"Successfully processed data for experiment '{item_name}'")

    # --- Create DataFrame ---
    if not all_results_data:
        logging.warning("No valid experiment data found to create a DataFrame.")
        return pd.DataFrame()

    try:
        results_df = pd.DataFrame(all_results_data)
        logging.info(f"Successfully created DataFrame with {len(results_df)} rows.")

        # Optional: Reorder columns for better readability
        id_col = ['experiment_id']
        config_cols = list(data.get('experiment_config', {}).keys())
        summary_cols = list(k for k in data.get('summary', {}).keys() if k not in ['global_results', 'client_results']) # Get summary keys added
        # Filter out columns that might not exist in all rows if parsing failed partially
        existing_cols = [col for col in id_col + config_cols + summary_cols if col in results_df.columns]
        results_df = results_df[existing_cols]

        # Optional: Set experiment_id as the DataFrame index
        # results_df.set_index('experiment_id', inplace=True)

    except Exception as e:
        logging.error(f"Failed to create DataFrame from processed data: {e}")
        return pd.DataFrame()


    return results_df

# --- Example Usage ---
# Note: Replace 'Central_results2' with the actual path to your folder.
# If running this script, make sure the folder exists relative to the script
# or provide an absolute path.

# Create dummy structure for demonstration if needed
def create_dummy_structure(base_dir="Central_results2_demo"):
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)
    # Sample data similar to the user's example
    experiments_data = {
        "Pubmed_adjacency_GCN": {'experiment_config': {'device': 'cuda', 'data_loading_option': 'adjacency', 'model_type': 'GCN', 'dataset': 'Pubmed', 'num_clients': 10, 'beta': 1, 'hop': 1, 'fulltraining_flag': False}, 'rounds': [{'round': 1, 'global_result': 0.715, 'client_result': 0.724}, {'round': 2, 'global_result': 0.67, 'client_result': 0.698}], 'summary': {'global_results': [0.715, 0.67], 'client_results': [0.724, 0.698], 'average_global_result': 0.6925, 'average_client_result': 0.711, 'std_global': 0.0225, 'std_client': 0.013}},
        "Pubmed_diffusion_GCN": {'experiment_config': {'device': 'cuda', 'data_loading_option': 'diffusion', 'model_type': 'GCN', 'dataset': 'Pubmed', 'num_clients': 10, 'beta': 1, 'hop': 2, 'fulltraining_flag': False}, 'rounds': [], 'summary': {'average_global_result': 0.701, 'average_client_result': 0.715, 'std_global': 0.021, 'std_client': 0.012}},
        "Cora_full_SAGE": {'experiment_config': {'device': 'cpu', 'data_loading_option': 'full', 'model_type': 'SAGE', 'dataset': 'Cora', 'num_clients': 1, 'beta': 0, 'hop': 0, 'fulltraining_flag': True}, 'rounds': [], 'summary': {'average_global_result': 0.81, 'average_client_result': 0.81, 'std_global': 0.0, 'std_client': 0.0}},
        "Folder_With_Invalid_Txt": "This is not a dictionary string", # Test invalid content
        "Folder_With_No_Txt": {}, # Test missing txt file
        "Folder_With_Missing_Keys": {'experiment_config': {'device': 'cpu'}, 'summary': {}} # Test missing keys
    }
    logging.info(f"Creating dummy structure in '{base_dir}'...")
    for folder_name, data in experiments_data.items():
        subfolder_path = os.path.join(base_dir, folder_name)
        if not os.path.exists(subfolder_path):
            os.makedirs(subfolder_path)

        if folder_name == "Folder_With_No_Txt":
            # Create an empty file or a file with a different extension
            with open(os.path.join(subfolder_path, "other_file.log"), 'w') as f:
                f.write("Log data")
            continue # Don't create the specific .txt file

        # Create the results text file
        txt_filename = f"results_{folder_name}.txt"
        txt_filepath = os.path.join(subfolder_path, txt_filename)
        try:
            with open(txt_filepath, 'w', encoding='utf-8') as f:
                if isinstance(data, dict):
                    # Convert dict to string representation for the file
                    f.write(str(data))
                else:
                    # Write invalid data directly
                    f.write(data)
        except IOError as e:
            logging.error(f"Failed to write dummy file {txt_filepath}: {e}")
    logging.info("Dummy structure created.")


# --- Main execution block ---
if __name__ == "__main__":
    # Create dummy data for testing
    dummy_folder_path = "Central_results2_demo"
    create_dummy_structure(dummy_folder_path)

    # Process the results folder
    results_dataframe = process_results_folder(dummy_folder_path)

    # Display the results
    if not results_dataframe.empty:
        print("\n--- Processed Results DataFrame ---")
        # Configure pandas display options for better readability
        pd.set_option('display.max_columns', None) # Show all columns
        pd.set_option('display.width', 1000)       # Adjust width
        print(results_dataframe)
    else:
        print("\nNo DataFrame was generated. Check log messages for details.")

    # Clean up dummy structure (optional)
    # import shutil
    # try:
    #     shutil.rmtree(dummy_folder_path)
    #     logging.info(f"Cleaned up dummy directory: '{dummy_folder_path}'")
    # except OSError as e:
    #     logging.error(f"Error removing dummy directory '{dummy_folder_path}': {e}")


2025-05-02 17:02:24,900 - INFO - Creating dummy structure in 'Central_results2_demo'...
2025-05-02 17:02:24,902 - INFO - Dummy structure created.
2025-05-02 17:02:24,902 - INFO - Starting processing for folder: 'Central_results2_demo'
2025-05-02 17:02:24,902 - INFO - Processing subfolder: 'Folder_With_Missing_Keys'
2025-05-02 17:02:24,903 - WARNING - Missing 'experiment_config' or 'summary' key in 'Central_results2_demo/Folder_With_Missing_Keys/results_Folder_With_Missing_Keys.txt'. Skipping file.
2025-05-02 17:02:24,903 - INFO - Processing subfolder: 'Folder_With_No_Txt'
2025-05-02 17:02:24,904 - WARNING - No .txt file found in subfolder 'Folder_With_No_Txt'. Skipping.
2025-05-02 17:02:24,904 - INFO - Processing subfolder: 'Folder_With_Invalid_Txt'
2025-05-02 17:02:24,905 - WARNING - Error parsing content in file 'Central_results2_demo/Folder_With_Invalid_Txt/results_Folder_With_Invalid_Txt.txt': invalid syntax (<unknown>, line 1). Skipping.
2025-05-02 17:02:24,905 - INFO - Processing


--- Processed Results DataFrame ---
          experiment_id device data_loading_option model_type dataset  num_clients  beta  hop  fulltraining_flag  average_global_result  average_client_result  std_global  std_client
0        Cora_full_SAGE    cpu                full       SAGE    Cora            1     0    0               True                 0.8100                  0.810      0.0000       0.000
1  Pubmed_adjacency_GCN   cuda           adjacency        GCN  Pubmed           10     1    1              False                 0.6925                  0.711      0.0225       0.013
2  Pubmed_diffusion_GCN   cuda           diffusion        GCN  Pubmed           10     1    2              False                 0.7010                  0.715      0.0210       0.012


In [11]:
results_folder = "Central_results474"
df = process_results_folder(results_folder)

2025-05-02 17:02:41,152 - INFO - Starting processing for folder: 'Central_results454'
2025-05-02 17:02:41,153 - INFO - Processing subfolder: 'Cora_adjacency_GCN'
2025-05-02 17:02:41,153 - INFO - Processing subfolder: 'Cora_full_GCN'
2025-05-02 17:02:41,154 - INFO - Processing subfolder: 'Cora_diffusion_GCN'
2025-05-02 17:02:41,155 - INFO - Successfully created DataFrame with 3 rows.


In [12]:
df

,experiment_id,device,data_loading_option,model_type,dataset,num_clients,beta,hop,fulltraining_flag,average_global_result,average_client_result,std_global,std_client
0,Cora_adjacency_GCN,cuda,adjacency,GCN,Cora,10,10000,1,False,0.616,0.359645,0.0,0.0
1,Cora_full_GCN,cuda,full,GCN,Cora,10,10000,1,False,0.627,0.357656,0.0,0.0
2,Cora_diffusion_GCN,cuda,diffusion,GCN,Cora,10,10000,1,False,0.596,0.301805,0.0,0.0


In [ ]:
results_folder2 = "Central_results3"
df2 = process_results_folder(results_folder2)
df2

In [ ]:
results_folder3 = "Central_results4"
df3 = process_results_folder(results_folder3)
df3